# ReBRAC Broad Validation v2 — Core Training Driver (Colab)

> 文档对应：[`docs/rebrac_broad_validation_v2_plan.md`](../docs/rebrac_broad_validation_v2_plan.md) (rev.3 + cmd fix, commits `819ed3e` / `79294b8` / `7a4bc2a`)
>
> 目的：跑完 plan rev.3 §6.2 的 **N0 (reward bridge)** + **N2' (oracle-teacher critical-regime)** 共 4 runs，应用 §5.2 / §5.3 verdict gate 决定是否触发 conditional M1 BC sweep。

## 执行摘要（首轮 2-seed Colab pass）

| Cell | Dataset | Manifest | Regime | Seeds | 用途 |
|---|---|---|---|---|---|
| **N0** | `crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000` | `single_u10_cross_tgt15` | sub-critical (U=1.0/Re=150) | 42, 0 | reward bridge anchor (vs main-line `efficiency_v2` 0.902 ± 0.021) |
| **N2'** | `privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000` | `single_u15_cross_tgt15` | critical (U=1.5/Re=250) | 42, 0 | oracle-teacher partial-obs ceiling probe (vs online catastrophic 0.10) |

**Pre-train phase (DONE locally on 2026-05-18, commits `819ed3e`/`79294b8`/`7a4bc2a`)**：
- S sanity: crosscomp = 0.0% (catastrophic) / privileged = 70.0% (ceiling) → rev.3 pivot 到 privileged collector
- N0 + N2' datasets 已 collect (CPU, 6.5 min total) + sanity verified
- Local CPU smoke test: N0 / seed=42 / num-epochs=2 → all ReBRAC losses converge (critic 120→27, actor 4.5→0.4, bc 0.9→0.05)

**预算**：2 cell × 2 seed × ~30–45 min L4 = **~2–3h L4** (best case, no M1 triggered).

**Verdict gates** (pre-registered，不允许 post-hoc 调整)：

| Cell | Gate | 触发动作 |
|---|---|---|
| N0 | mean ≥ 0.70 → proceed N2' / 0.50–0.70 → weak / < 0.50 → **暂停所有后续** | §5.2 |
| N2' | mean ≥ 0.40 strong positive / [0.15, 0.40] partial → trigger M1 / < 0.15 strong negative | §5.3 |
| M1 | conditional on N2' ∈ [0.15, 0.40]，β1 ∈ {0,1,2,4,8} × 3 seed = 15 runs ≈ 15h L4 | 本 notebook 不跑，另起 |

**Important caveat (rev.3 §2.4)**：privileged dataset 含 priv-action，但 ReBRAC actor 仍只看 s0_obs，学到的是 `E[priv_action | s0_obs]`，**不是** priv_action 本身。因此 N2' 数字封顶是 actor-fundamental ceiling under s0，不会真的达到 70%。

## 0. 环境 sanity

In [ ]:
!lscpu | head -8
print()
!nvidia-smi

Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  12
On-line CPU(s) list:                     0-11
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz

Mon May 18 15:24:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |              

In [ ]:
import torch
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device 0      : {torch.cuda.get_device_name(0)}")
    print(f"cuDNN         : {torch.backends.cudnn.version()}")

PyTorch       : 2.10.0+cu128
CUDA available: True
Device 0      : NVIDIA L4
cuDNN         : 91002


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


In [ ]:
!pwd && ls scripts/train_offline.py docs/rebrac_broad_validation_v2_plan.md
!ls offline_data/crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/transitions.npz \
    offline_data/privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000/transitions.npz
!ls benchmarks/single_u10_cross_tgt15.json benchmarks/single_u15_cross_tgt15.json

/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5
docs/rebrac_broad_validation_v2_plan.md  scripts/train_offline.py
offline_data/crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/transitions.npz
offline_data/privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000/transitions.npz
benchmarks/single_u10_cross_tgt15.json	benchmarks/single_u15_cross_tgt15.json


## 1. Run matrix

4 runs = 2 cell × 2 seed。每个 run 都是同一个 canonical `train_offline.py` invocation，只换 `--offline-data` / `--manifest` / `--seed` / `--save-dir`（plan rev.3 §6.2，bug-fixed in commit `7a4bc2a`）。

Skip-resume 通过同时检查 `trainer_state.json` + `agent_final.pt` 存在性实现 — 如果都齐全说明上一次跑完整，跳过；否则重跑（`train_offline.py` 内部用 `maybe_resume()` 支持断点续训）。

In [ ]:
from pathlib import Path

REPO_ROOT = Path('.').resolve()
CKPT_ROOT = REPO_ROOT / 'checkpoints' / 'offline' / 'rebrac' / 'broad_validation_v2'
RESULT_ROOT = REPO_ROOT / 'experiments' / 'offline' / 'rebrac' / 'broad_validation_v2'

CELLS = {
    'N0': {
        'dataset': 'offline_data/crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/transitions.npz',
        'manifest': 'benchmarks/single_u10_cross_tgt15.json',
        'regime_note': 'sub-critical (U=1.0/Re=150)',
    },
    'N2p': {
        'dataset': 'offline_data/privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000/transitions.npz',
        'manifest': 'benchmarks/single_u15_cross_tgt15.json',
        'regime_note': 'critical (U=1.5/Re=250)',
    },
}

SEEDS = [42, 0]

RUNS = []
for cell_id, cfg in CELLS.items():
    for seed in SEEDS:
        save_dir = CKPT_ROOT / cell_id / f'seed_{seed}'
        RUNS.append({
            'cell_id': cell_id,
            'seed': seed,
            'dataset': cfg['dataset'],
            'manifest': cfg['manifest'],
            'save_dir': str(save_dir),
            'regime_note': cfg['regime_note'],
        })

print(f'{"#":>2} {"cell":<5} {"seed":>5} {"save_dir":<60}')
print('-' * 80)
for i, r in enumerate(RUNS, 1):
    print(f'{i:>2} {r["cell_id"]:<5} {r["seed"]:>5} {r["save_dir"]}')

 # cell   seed save_dir                                                    
--------------------------------------------------------------------------------
 1 N0       42 /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/checkpoints/offline/rebrac/broad_validation_v2/N0/seed_42
 2 N0        0 /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/checkpoints/offline/rebrac/broad_validation_v2/N0/seed_0
 3 N2p      42 /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/checkpoints/offline/rebrac/broad_validation_v2/N2p/seed_42
 4 N2p       0 /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/checkpoints/offline/rebrac/broad_validation_v2/N2p/seed_0


## 2. Training (4 runs, ~2–3h L4 total)

Canonical command (plan rev.3 §6.2, post-fix `7a4bc2a`):

- `--algo rebrac` (注意：是 `scripts.train_offline` 不是 `scripts.train_offline_rebrac`)
- `--actor-penalty-coef 4.0 --critic-penalty-coef 2.0`（不是 `--actor-bc/--critic-bc`）
- `--sampling-mode shuffle_no_replacement --num-epochs 64`（epoch-based，不是 `--epochs`）
- `--manifest`（不是 `--eval-manifest`）
- `--critic-layernorm --no-actor-layernorm`
- `--eval-every 0 --skip-final-eval`（评估在 §3 separately run，加速训练）

In [ ]:
import subprocess
import time

for i, r in enumerate(RUNS, 1):
    save_dir = Path(r['save_dir'])
    trainer_state = save_dir / 'trainer_state.json'
    agent_final = save_dir / 'agent_final.pt'

    print(f'\n========== [{i}/{len(RUNS)}] {r["cell_id"]} seed={r["seed"]} ({r["regime_note"]}) ==========')

    if trainer_state.exists() and agent_final.exists():
        print(f'[skip] already complete: {save_dir}')
        continue

    save_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    cmd = [
        'python', '-m', 'scripts.train_offline',
        '--algo', 'rebrac',
        '--offline-data', r['dataset'],
        '--manifest', r['manifest'],
        '--probe-layout', 's0',
        '--history-length', '4',
        '--task-geometry', 'cross_stream',
        '--target-speed', '1.5',
        '--objective', 'arrival_v2',
        '--sampling-mode', 'shuffle_no_replacement',
        '--num-epochs', '64',
        '--batch-size', '256',
        '--hidden-dim', '256',
        '--num-hidden-layers', '3',
        '--actor-lr', '3e-4',
        '--critic-lr', '3e-4',
        '--gamma', '0.99',
        '--tau', '0.005',
        '--actor-penalty-coef', '4.0',
        '--critic-penalty-coef', '2.0',
        '--policy-noise', '0.2',
        '--noise-clip', '0.5',
        '--policy-freq', '2',
        '--grad-clip-norm', '10.0',
        '--normalizer-eps', '1e-3',
        '--critic-layernorm',
        '--no-actor-layernorm',
        '--eval-every', '0',
        '--skip-final-eval',
        '--log-every', '1000',
        '--seed', str(r['seed']),
        '--device', 'cuda',
        '--save-dir', str(save_dir),
    ]
    print(' '.join(cmd))
    result = subprocess.run(cmd)
    elapsed = (time.time() - t0) / 60
    if result.returncode != 0:
        print(f'\n[FAIL] {r["cell_id"]} seed={r["seed"]} exit={result.returncode} ({elapsed:.1f} min)')
        break
    print(f'[done] {r["cell_id"]} seed={r["seed"]} → {save_dir} ({elapsed:.1f} min)')


========== [1/4] N0 seed=42 (sub-critical (U=1.0/Re=150)) ==========
python -m scripts.train_offline --algo rebrac --offline-data offline_data/crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/transitions.npz --manifest benchmarks/single_u10_cross_tgt15.json --probe-layout s0 --history-length 4 --task-geometry cross_stream --target-speed 1.5 --objective arrival_v2 --sampling-mode shuffle_no_replacement --num-epochs 64 --batch-size 256 --hidden-dim 256 --num-hidden-layers 3 --actor-lr 3e-4 --critic-lr 3e-4 --gamma 0.99 --tau 0.005 --actor-penalty-coef 4.0 --critic-penalty-coef 2.0 --policy-noise 0.2 --noise-clip 0.5 --policy-freq 2 --grad-clip-norm 10.0 --normalizer-eps 1e-3 --critic-layernorm --no-actor-layernorm --eval-every 0 --skip-final-eval --log-every 1000 --seed 42 --device cuda --save-dir /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/checkpoints/offline/rebrac/broad_validation_v2/N0/seed_42
[done] N0 seed=42 → /content/drive/MyDrive/Colab Notebooks/new_offRL/

## 3. Evaluation (4 runs × 100 episodes against fixed manifest)

训练时关掉了 `--eval-every`，这里 separately 跑 `scripts.evaluate_offline` 对每个 checkpoint 走完整 100 ep manifest，结果写 `test_result.json`。Skip-resume：检查 `test_result.json` 存在性。

In [ ]:
for i, r in enumerate(RUNS, 1):
    save_dir = Path(r['save_dir'])
    test_json = save_dir / 'test_result.json'

    print(f'\n========== [{i}/{len(RUNS)}] eval {r["cell_id"]} seed={r["seed"]} ==========')

    if test_json.exists():
        print(f'[skip] test_result.json exists: {test_json}')
        continue
    if not (save_dir / 'agent_final.pt').exists():
        print(f'[skip] no agent_final.pt at {save_dir} (training not done?)')
        continue

    cmd = [
        'python', '-m', 'scripts.evaluate_offline',
        '--checkpoint', str(save_dir),
        '--agent-file', 'agent_final.pt',
        '--manifest', r['manifest'],
        '--episodes', '100',
        '--seed', '123',
        '--device', 'cuda',
        '--num-workers', '4',
        '--worker-device', 'cpu',
        '--output-json', str(test_json),
    ]
    print(' '.join(cmd))
    result = subprocess.run(cmd)
    if result.returncode != 0:
        print(f'[FAIL] {r["cell_id"]} seed={r["seed"]} eval exit={result.returncode}')
        break
    print(f'[done] {test_json}')


========== [1/4] eval N0 seed=42 ==========
python -m scripts.evaluate_offline --checkpoint /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/checkpoints/offline/rebrac/broad_validation_v2/N0/seed_42 --agent-file agent_final.pt --manifest benchmarks/single_u10_cross_tgt15.json --episodes 100 --seed 123 --device cuda --num-workers 4 --worker-device cpu --output-json /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/checkpoints/offline/rebrac/broad_validation_v2/N0/seed_42/test_result.json
[done] /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/checkpoints/offline/rebrac/broad_validation_v2/N0/seed_42/test_result.json

========== [2/4] eval N0 seed=0 ==========
python -m scripts.evaluate_offline --checkpoint /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/checkpoints/offline/rebrac/broad_validation_v2/N0/seed_0 --agent-file agent_final.pt --manifest benchmarks/single_u10_cross_tgt15.json --episodes 100 --seed 123 --device cuda --num-workers 4 --worker-devi

## 4. Summary + verdict gate

汇总每个 cell 的 2-seed (mean, std)，应用 §5.2 / §5.3 verdict gate。

In [ ]:
import json
import statistics

ANCHOR_EFF_V2_MEAN = 0.902  # main-line efficiency_v2 5-seed anchor
ONLINE_CATASTROPHIC = 0.10  # online §7.6 single_cross_s0 floor
ORACLE_CEILING = 0.70  # privileged S sanity ceiling

summary = {}
for r in RUNS:
    test_json = Path(r['save_dir']) / 'test_result.json'
    if not test_json.exists():
        print(f'[missing] {test_json}')
        continue
    d = json.loads(test_json.read_text())
    succ = d.get('eval_success_rate') or d.get('success_rate')
    summary.setdefault(r['cell_id'], []).append({'seed': r['seed'], 'success': succ, 'raw': d})

print('=' * 88)
print(f'{"cell":<6}{"n":>4}{"mean":>9}{"std":>9}{"seeds":>22}{"per-seed succ":>26}')
print('-' * 88)
rows = {}
for cell_id, results in summary.items():
    succ_list = [x['success'] for x in results if x['success'] is not None]
    if not succ_list:
        continue
    mean = statistics.mean(succ_list)
    std = statistics.stdev(succ_list) if len(succ_list) > 1 else 0.0
    seeds = ','.join(str(x['seed']) for x in results)
    per_seed = ' '.join(f'{x["success"]:.3f}' for x in results)
    rows[cell_id] = {'mean': mean, 'std': std, 'n': len(succ_list)}
    print(f'{cell_id:<6}{len(succ_list):>4}{mean:>9.4f}{std:>9.4f}{seeds:>22}{per_seed:>26}')
print('=' * 88)

print('\n--- §5.2 N0 verdict (reward bridge) ---')
if 'N0' in rows:
    n0 = rows['N0']['mean']
    print(f'  N0 mean = {n0:.4f} (vs main-line efficiency_v2 anchor {ANCHOR_EFF_V2_MEAN:.3f})')
    if n0 >= 0.70:
        print('  → reward bridge HOLDS — proceed to N2\' analysis')
    elif n0 >= 0.50:
        print('  → WEAK bridge — N2\' 仍跑，但 paper 写作时标明 arrival_v2 退化 X pp')
    else:
        print('  → reward bridge FAILS — 暂停所有后续 cell，review reward/collector/dataset')

print('\n--- §5.3 N2\' verdict (oracle-teacher partial-obs ceiling) ---')
if 'N2p' in rows:
    n2p = rows['N2p']['mean']
    print(f'  N2\' mean = {n2p:.4f} (vs online floor {ONLINE_CATASTROPHIC:.2f}, oracle ceiling {ORACLE_CEILING:.2f})')
    if n2p >= 0.40:
        print('  → STRONG POSITIVE — offline RL with oracle demos meaningfully bridges partial-obs gap')
        print('    Paper claim: s0-conditioned imitation extracts ≥ half of oracle ceiling')
    elif n2p >= 0.15:
        print(f'  → PARTIAL — recovers {(n2p/ORACLE_CEILING)*100:.1f}% of oracle teacher')
        print('    Trigger M1 BC penalty sweep (β1 ∈ {0,1,2,4,8} × 3 seed = 15 runs, ~15h L4)')
    else:
        print('  → STRONG NEGATIVE — critical regime is actor-fundamental under s0')
        print('    Paper claim: even oracle demos cannot bridge partial-obs gap when actor lacks hull-integral flow')

print('\n--- 2-seed limitation reminder ---')
print('  本轮只跑 2 seed [42, 0]，std df=1 极不稳。若 N2\' 落入 partial zone 或论文审稿要 3-seed power，')
print('  补 seed 43 后重跑 §3 + §4，得到 spec 完整 3-seed verdict。')

cell     n     mean      std                 seeds             per-seed succ
----------------------------------------------------------------------------------------
N0       2   0.8500   0.0236                  42,0               0.867 0.833

--- §5.2 N0 verdict (reward bridge) ---
  N0 mean = 0.8500 (vs main-line efficiency_v2 anchor 0.902)
  → reward bridge HOLDS — proceed to N2' analysis

--- §5.3 N2' verdict (oracle-teacher partial-obs ceiling) ---

--- 2-seed limitation reminder ---
  本轮只跑 2 seed [42, 0]，std df=1 极不稳。若 N2' 落入 partial zone 或论文审稿要 3-seed power，
  补 seed 43 后重跑 §3 + §4，得到 spec 完整 3-seed verdict。


## 5. 跑完后清单（回到 local）

1. **sync back** `checkpoints/offline/rebrac/broad_validation_v2/{N0,N2p}/seed_{42,0}/` 的 `test_result.json` + `trainer_state.json`（agent_final.pt 可选，大文件）到 git 仓库
2. **更新** [`docs/rebrac_broad_validation_v2_plan.md`](../docs/rebrac_broad_validation_v2_plan.md) §10 status：填入实测 2-seed 数字 + verdict 决定
3. **新建** `docs/rebrac_broad_validation_v2_report.md`：N0 / N2' 数字 + §5.2 / §5.3 verdict 应用结果 + 是否触发 M1
4. **若 N2' partial**：另起 notebook 跑 M1 BC sweep（15 runs）
5. **若 N2' strong negative**：直接落 actor-fundamental ceiling 论断，broad val v2 收口